# SQD Solver with IBM Quantum Backend

This notebook demonstrates how to use the SQD (Sample-based Quantum Diagonalization) solver with IBM Quantum hardware.

## Overview

The SQD algorithm combines:
1. **Quantum Sampling**: Build and execute LUCJ ansatz on quantum hardware
2. **Classical Post-Processing**: Diagonalize subspaces using SBD solver

## Prerequisites

Before running this notebook:
1. Create `qpu_config.yaml` from the template
2. Add your IBM Quantum credentials
3. Ensure SBD solver is installed (for post-processing)

## Setup

In [9]:
import numpy as np
from pyscf import gto, scf, cc, ao2mo

# Import quantum fragment methods
from quantum_fragment_methods.qpu import IBMQuantumBackend
from quantum_fragment_methods.application.solvers.quantum_zoo.sqd import SQDSolver
from quantum_fragment_methods.config import load_qpu_config

## 1. Define Molecular System

We'll use a simple H2 molecule as an example.

In [10]:
# Define H2 molecule
mol = gto.Mole()
mol.atom = '''
H 0 0 0
H 0 0 0.74
'''
mol.basis = 'sto-3g'
mol.build()

print(f"Number of orbitals: {mol.nao}")
print(f"Number of electrons: {mol.nelectron}")

Number of orbitals: 2
Number of electrons: 2


## 2. Run Mean-Field Calculation

Perform Hartree-Fock to get initial guess.

In [11]:
# Hartree-Fock calculation
mf = scf.RHF(mol)
mf.kernel()

print(f"HF energy: {mf.e_tot:.8f}")

converged SCF energy = -1.11675930739643
HF energy: -1.11675931


## 3. Get Hamiltonian Integrals

In [12]:
# Extract Hamiltonian components
norb = mol.nao
nelec = (mol.nelectron // 2, mol.nelectron // 2)

h1e = mf.get_hcore()
h2e = ao2mo.restore(1, mf._eri, norb)

print(f"One-body tensor shape: {h1e.shape}")
print(f"Two-body tensor shape: {h2e.shape}")

One-body tensor shape: (2, 2)
Two-body tensor shape: (2, 2, 2, 2)


## 4. Load Configuration

Load QPU and SQD configuration from `config.yaml`.

The config file contains:
- **qpu**: IBM Quantum backend settings and credentials
- **sqd**: SQD solver parameters (LUCJ, transpilation, algorithm settings, SBD)
- **ext_sqd**: Extended SQD configuration

In [13]:
import yaml
from pathlib import Path

# Load configuration from config.yaml in the same directory
# For container workflows, this would be mounted at /workspace/config.yaml
config_path: Path = Path('/workspace/quantum-fragment-methods/examples/local_demo/config.yaml')

# Load configuration
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Extract configurations
qpu_config = config['qpu']
sqd_config = config['sqd']

print(f"✓ Configuration loaded from: {config_path.absolute()}")
print(f"\nQPU Settings:")
print(f"  Provider: {qpu_config['provider']}")
print(f"  Backend: {qpu_config['backend_name']}")
print(f"  Shots: {qpu_config['sampler_options']['default_shots']:,}")
print(f"\nSQD Settings:")
print(f"  Iterations: {sqd_config['iterations']}")
print(f"  Batches: {sqd_config['n_batches']}")
print(f"  Samples per batch: {sqd_config['samples_per_batch']:,}")

✓ Configuration loaded from: /workspace/quantum-fragment-methods/examples/local_demo/config.yaml

QPU Settings:
  Provider: ibm_quantum
  Backend: ibm_pittsburgh
  Shots: 1,000,000

SQD Settings:
  Iterations: 5
  Batches: 10
  Samples per batch: 3,000


## 5. Initialize IBM Quantum Backend

Connect to IBM Quantum hardware using the configuration.

In [14]:
# Initialize backend (only run after adding credentials above)
backend = IBMQuantumBackend(qpu_config)
backend.initialize()
backend.get_backend()

# Display backend properties
props = backend.get_backend_properties()
print(f"Backend: {props['backend_name']}")
print(f"Number of qubits: {props['num_qubits']}")
print(f"Basis gates: {props['basis_gates']}")

qiskit_runtime_service._discover_account:WARNING:2026-04-02 03:27:20,177: Loading account with the given token. A saved account will not be used.


Backend: ibm_pittsburgh
Number of qubits: 156
Basis gates: ['cz', 'id', 'rz', 'sx', 'x']


## 5. Configure SQD Solver

In [15]:
# Extract SQD solver configuration from config.yaml
sqd_config = config['sqd']

print("SQD Configuration:")
print(f"  Iterations: {sqd_config['iterations']}")
print(f"  Batches: {sqd_config['n_batches']}")
print(f"  Samples per batch: {sqd_config['samples_per_batch']}")

# Create SQD solver
solver = SQDSolver(backend, config=sqd_config)
print(f"\nInitialized {solver.name} solver")

SQD Configuration:
  Iterations: 5
  Batches: 10
  Samples per batch: 3000

Initialized SQD solver


## 6. Run SQD Calculation

This will:
1. Build LUCJ circuit from CCSD amplitudes
2. Submit to IBM Quantum hardware
3. Retrieve measurement counts
4. Run SBD post-processing

In [18]:
# Run SQD solver
# Note: This will submit a job to IBM Quantum and may take time to complete

result = solver.solve(
    h1e=h1e,
    h2e=h2e,
    norb=norb,
    nelec=nelec,
    mf=mf,  # Will compute CCSD amplitudes automatically
    workflow_path='./sqd_results',
    force_resubmit=False
)

print(f"\nSQD Results:")
print(f"Energy: {result.energy:.8f}")
print(f"Metadata: {result.metadata}")

Iteration 1: Handling 10 batches
####################
Processing batch 1/10...
Processing batch 2/10...
Processing batch 3/10...
Processing batch 4/10...
Processing batch 5/10...
Processing batch 6/10...
Processing batch 7/10...
Processing batch 8/10...
Processing batch 9/10...
Processing batch 10/10...
####################
Completed batches
####################
Iteration 2: Handling 10 batches
####################
Processing batch 1/10...
Processing batch 2/10...
Processing batch 3/10...
Processing batch 4/10...
Processing batch 5/10...
Processing batch 6/10...
Processing batch 7/10...
Processing batch 8/10...
Processing batch 9/10...
Processing batch 10/10...
####################
Completed batches
####################
Energy change: 0.00e+00, Occupancy change: 0.00e+00
Converged after 2 iterations

SQD Results:
Energy: -1.76490352
Metadata: {'ci_strs_a': [1, 2], 'ci_strs_b': [1, 2], 'occupancies': (array([0.5, 0.5]), array([0.5, 0.5])), 'norb': 2, 'nelec': (1, 1)}


## 7. CCSD Comparison

For validation, let's compare the SQD result with classical CCSD.

In [19]:
# Run CCSD calculation for comparison
mycc = cc.CCSD(mf)
mycc.kernel()

print(f"\nClassical Results:")
print(f"HF energy:   {mf.e_tot:.8f}")
print(f"CCSD energy: {mycc.e_tot:.8f}")
print(f"Correlation energy: {mycc.e_corr:.8f}")

print(f"\nSQD Results:")
print(f"SQD energy:  {result.energy:.8f}")

# Calculate errors
sqd_error = abs(result.energy - mycc.e_tot)
print(f"\nEnergy Comparison:")
print(f"SQD vs CCSD error: {sqd_error:.8f} Ha ({sqd_error * 627.5:.4f} kcal/mol)")

E(CCSD) = -1.13728399861044  E_corr = -0.02052469121401437

Classical Results:
HF energy:   -1.11675931
CCSD energy: -1.13728400
Correlation energy: -0.02052469

SQD Results:
SQD energy:  -1.76490352

Energy Comparison:
SQD vs CCSD error: 0.62761952 Ha (393.8312 kcal/mol)


## Summary

This notebook demonstrated:
- Setting up IBM Quantum backend with configuration
- Creating and configuring SQD solver
- Building LUCJ circuits from CCSD amplitudes
- Submitting jobs to quantum hardware
- Monitoring job status
- Comparing SQD results with classical CCSD for validation